In [ ]:
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt

from static_training import train_static
from progressive_training import train_progressive
from mistake_driven_training import train_mistake_driven

In [ ]:
# Load a single file first (or subset)
df = pd.read_csv("data/airlines/2010.csv")

df.head()

In [ ]:
# Drop useless column
df = df.drop(columns=['Unnamed: 27'], errors='ignore')

# Remove cancelled/diverted flights
df = df[df['CANCELLED'] == 0]
df = df[df['DIVERTED'] == 0]

# Remove rows that don't have an arrival delay value (Still keeps zero)
df = df[df['ARR_DELAY'].notna()]

In [ ]:
drop_cols = [
    'ACTUAL_ELAPSED_TIME',
    'ARR_DELAY',       # target
    'ARR_TIME',
    'TAXI_IN',
    'WHEELS_ON'
]

y = df['ARR_DELAY']
X = df.drop(columns=drop_cols + ['FL_DATE'])

One HotEncode Carrier, Destination, Origin

In [ ]:
# Encode OP_CARRIER (safe to one-hot)
X = pd.get_dummies(X, columns=['OP_CARRIER'], drop_first=False)

# Label encode ORIGIN and DEST (avoid huge one-hot)
from sklearn.preprocessing import LabelEncoder

le_origin = LabelEncoder()
le_dest = LabelEncoder()

X['ORIGIN'] = le_origin.fit_transform(X['ORIGIN'])
X['DEST'] = le_dest.fit_transform(X['DEST'])

In [ ]:
# Keep time order for your models
df = df.sort_values('FL_DATE')

X = X.loc[df.index]
y = y.loc[df.index]

In [ ]:
look_back_values = [5000, 10000, 20000]   # rows (~time window)
tree_depths = [3, 5, 7]

look_ahead = 1000       # prediction window
retrain_step = 5000     # progressive retraining frequency
threshold = 20          # mistake-driven RMSE trigger

# Static Model

In [ ]:
results_static = {}
time_static = {}

for lb in look_back_values:
    results_static[lb] = {}
    time_static[lb] = {}
    
    for depth in tree_depths:
        start_time = time.time()
        
        _, avg_rmse = train_static(X, y, look_back=lb, depthOfTree=depth)
        
        elapsed_time = time.time() - start_time
        
        results_static[lb][depth] = avg_rmse
        time_static[lb][depth] = elapsed_time
        
        print(f"[Static] LB {lb}, Depth {depth} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

# Progressive Model

In [ ]:
results_progressive = {}
time_progressive = {}

for lb in look_back_values:
    results_progressive[lb] = {}
    time_progressive[lb] = {}
    
    for depth in tree_depths:
        start_time = time.time()
        
        _, avg_rmse = train_progressive(
            X, y,
            look_back=lb,
            look_ahead=look_ahead,
            retrain_step=retrain_step,
            depthOfTree=depth
        )
        
        elapsed_time = time.time() - start_time
        
        results_progressive[lb][depth] = avg_rmse
        time_progressive[lb][depth] = elapsed_time
        
        print(f"[Progressive] LB {lb}, Depth {depth} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

#  Mistake Model

In [ ]:
results_mistake = {}
time_mistake = {}

for lb in look_back_values:
    results_mistake[lb] = {}
    time_mistake[lb] = {}
    
    for depth in tree_depths:
        start_time = time.time()
        
        _, avg_rmse = train_mistake_driven(
            X, y,
            look_back=lb,
            look_ahead=look_ahead,
            depthOfTree=depth,
            threshold=threshold
        )
        
        elapsed_time = time.time() - start_time
        
        results_mistake[lb][depth] = avg_rmse
        time_mistake[lb][depth] = elapsed_time
        
        print(f"[Mistake] LB {lb}, Depth {depth} → RMSE: {avg_rmse:.2f}, Time: {elapsed_time:.2f}s")

In [ ]:
df_static_rmse = pd.DataFrame(results_static).T
df_progressive_rmse = pd.DataFrame(results_progressive).T
df_mistake_rmse = pd.DataFrame(results_mistake).T

df_static_time = pd.DataFrame(time_static).T
df_progressive_time = pd.DataFrame(time_progressive).T
df_mistake_time = pd.DataFrame(time_mistake).T

def plot_three_heatmaps(dfs, titles, cmap, vmin, vmax):
    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    
    for i in range(3):
        sns.heatmap(dfs[i].astype(float), annot=True, fmt=".2f",
                    cmap=cmap, vmin=vmin, vmax=vmax, ax=axes[i])
        axes[i].set_title(titles[i])
        axes[i].set_xlabel("Tree Depth")
        axes[i].set_ylabel("Look-Back")
    
    plt.tight_layout()
    plt.show()

In [ ]:
rmse_min = min(df_static_rmse.min().min(),
               df_progressive_rmse.min().min(),
               df_mistake_rmse.min().min())

rmse_max = max(df_static_rmse.max().max(),
               df_progressive_rmse.max().max(),
               df_mistake_rmse.max().max())

plot_three_heatmaps(
    [df_static_rmse, df_progressive_rmse, df_mistake_rmse],
    ["Static RMSE", "Progressive RMSE", "Mistake RMSE"],
    "coolwarm",
    rmse_min,
    rmse_max
)

In [ ]:
time_min = min(df_static_time.min().min(),
               df_progressive_time.min().min(),
               df_mistake_time.min().min())

time_max = max(df_static_time.max().max(),
               df_progressive_time.max().max(),
               df_mistake_time.max().max())

plot_three_heatmaps(
    [df_static_time, df_progressive_time, df_mistake_time],
    ["Static Time", "Progressive Time", "Mistake Time"],
    "YlGnBu",
    time_min,
    time_max
)